In [3]:
import requests
import random
import os
import json
import csv
import logging
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt')


logging.basicConfig(level=logging.INFO)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/giacomomunda/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Wikidata

In [3]:
# Decreasing the size of the Wikidata5M dataset

file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train.txt'
output_file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_30k.tsv'
num_samples = 40_000

with open(file, 'r') as f:
    lines = f.readlines()

random_sample = random.sample(lines, num_samples)

with open(output_file, 'w') as f:
    f.writelines(random_sample)

## Exploratory Analysis

In [21]:
dataset = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

df = pd.read_csv(dataset, sep='\t', header=None)
df.columns = ['head', 'relation', 'tail']
num_relations = df['relation'].unique()
print(f"Number of relations: {len(num_relations)}")

relations_counts = df['relation'].value_counts()
print(f"Value counts of relation: {relations_counts}")

# choosing relations with more than 50 entities 
relations = relations_counts[relations_counts > 50].index.tolist()
print(f"Number of relations after filtering: {len(relations)}")

Number of relations: 50
Value counts of relation: relation
P31      8392
P17      3005
P27      2507
P106     2406
P131     2005
P54      2005
P19      1867
P735     1832
P161     1113
P641     1062
P69       960
P47       906
P421      882
P105      817
P136      806
P171      750
P20       609
P495      569
P1412     521
P1344     486
P166      398
P175      396
P413      396
P264      334
P155      320
P156      305
P364      297
P361      295
P102      276
P734      235
P150      235
P57       215
P463      210
P279      208
P407      205
P159      180
P108      175
P39       165
P3373     164
P937      156
P607      153
P360      144
P527      143
P40       140
P86       136
P162      128
P50       125
P137      122
P141      113
P22       108
Name: count, dtype: int64
Number of relations after filtering: 50


In [24]:
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

# Initialize a dictionary to count triples for each relation
relation_counts = defaultdict(int)

# Load and analyze the dataset
with open(file_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            # Assuming the format is: head_entity, relation, tail_entity
            relation = parts[1]
            relation_counts[relation] += 1

# Sort the relations by their counts in descending order
sorted_relations = sorted(relation_counts.items(), key=lambda item: item[1], reverse=True)

# Print the total number of relations and some examples of counts
print(f"Total number of unique relations: {len(relation_counts)}")
for relation, count in sorted_relations[:10]:
    print(f"Relation: {relation}, Count: {count}")

Total number of unique relations: 200
Relation: P31, Count: 7547
Relation: P17, Count: 2702
Relation: P27, Count: 2254
Relation: P106, Count: 2164
Relation: P131, Count: 1803
Relation: P54, Count: 1803
Relation: P19, Count: 1679
Relation: P735, Count: 1648
Relation: P161, Count: 1001
Relation: P641, Count: 955


## Reducing the dataset size through proportional sampling

In [2]:
def proportional_sample(file_path, num_samples, target_num_relations, min_samples_per_relation=10):
    relation_counts = defaultdict(int)
    relation_triples = defaultdict(list)

    # First pass: count occurrences and collect triples
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                relation = parts[1]
                relation_counts[relation] += 1
                relation_triples[relation].append(line)
    
    # Select the top N relations by count
    top_relations = sorted(relation_counts, key=relation_counts.get, reverse=True)[:target_num_relations]

    # Adjusted total count to only consider top relations
    total_count = sum(relation_counts[relation] for relation in top_relations)
    
    # Proportional sampling within the top relations
    sampled_triples = []
    for relation in top_relations:
        proportion = relation_counts[relation] / total_count
        samples_for_relation = max(int(proportion * num_samples), min_samples_per_relation)
        
        # Ensure not to exceed the actual number of available triples
        samples_for_relation = min(samples_for_relation, len(relation_triples[relation]))
        
        sampled_triples.extend(random.sample(relation_triples[relation], samples_for_relation))
    
    return sampled_triples

def write_to_file(output_file, sampled_triples):
    with open(output_file, 'w') as f:
        f.writelines(sampled_triples)

# Example usage parameters
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train.txt'  # Update this to your actual file path
output_file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k.tsv'   # Desired output file path
num_samples = 300_000  # Target number of samples in the reduced dataset
target_num_relations = 200  # Target number of relations to keep
min_samples_per_relation = 50  # Minimum samples per relation to ensure representation

# Execute the sampling
sampled_triples = proportional_sample(file_path, num_samples, target_num_relations, min_samples_per_relation)
write_to_file(output_file, sampled_triples)


In [4]:
def convert_tsv_to_jsonl(tsv_file_path, jsonl_file_path):
    """
    Converts a dataset from TSV format to JSONL format.
    
    Parameters:
    - tsv_file_path: str. The path to the input TSV file.
    - jsonl_file_path: str. The path to the output JSONL file.
    """
    with open(tsv_file_path, 'r') as tsv_file, open(jsonl_file_path, 'w') as jsonl_file:
        for line in tsv_file:
            sub_id, pred_id, obj_id = line.strip().split('\t')
            data = {"sub_id": sub_id, "pred_id": pred_id, "obj_id": obj_id}
            jsonl_file.write(json.dumps(data) + '\n')

if __name__ == "__main__":
    tsv_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k.tsv'
    jsonl_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k.jsonl'
    convert_tsv_to_jsonl(tsv_file_path, jsonl_file_path)

## Wikipedia Descriptions

TODO:

1. Delete all entities that are not in the final dataset

2. Take only the first 2 sentences for each entity

In [4]:
# 1. Delete all entities that are not in the final dataset

def filter_descriptions(triples_path, descriptions_path, output_path):
    # Step 1: Read the Wikidata triples and extract unique entity IDs
    entity_ids = set()
    with open(triples_path, 'r') as triples_file:
        for line in triples_file:
            triple = json.loads(line.strip())  # Parse JSON line
            entity_ids.add(triple["sub_id"])  # Add subject entity ID
            entity_ids.add(triple["obj_id"])  # Add object entity ID

    # Step 2: Filter the Wikipedia descriptions
    filtered_descriptions = []
    with open(descriptions_path, 'r') as descriptions_file:
        for line in descriptions_file:
            entity_id = line.strip().split('\t')[0]
            if entity_id in entity_ids:
                filtered_descriptions.append(line)

    # Step 3: Save the filtered descriptions to a new file
    with open(output_path, 'w') as output_file:
        for description in filtered_descriptions:
            output_file.write(description)

    print(f"Filtered descriptions saved to {output_path}")

triples_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc.jsonl' 
descriptions_path = '/Users/giacomomunda/Downloads/wikidata5m_text.txt'
output_path = './../../data/wikidata5m_descriptions/wikidata5m_text_filtered.tsv'

filter_descriptions(triples_path, descriptions_path, output_path)


Filtered descriptions saved to ./../../data/wikidata5m_descriptions/wikidata5m_text_filtered.tsv


**Keeping only the first two sentences**

In [6]:
with open('./../../data/wikidata5m_descriptions/wikidata5m_text_filtered.tsv', 'r') as file:
    dataset = file.read()

def keep_first_two_sentences_nltk(text):
    sentences = sent_tokenize(text)
    return ' '.join(sentences[:2])

processed_dataset = []
for line in dataset.strip().split('\n'):
    entity, description = line.split('\t', 1)
    cleaned_description = keep_first_two_sentences_nltk(description)
    processed_dataset.append(f"{entity}\t{cleaned_description}")

cleaned_dataset = '\n'.join(processed_dataset)

# Write the cleaned dataset to a file
with open('./../../data/wikidata5m_descriptions/wikidata5m_2sentences.tsv', 'w') as file:
    file.write(cleaned_dataset)

print("The processed dataset has been saved to processed_dataset_nltk.tsv.")

The processed dataset has been saved to processed_dataset_nltk.tsv.


**Append the descriptions to the final JSONL file**

In [3]:
# Load TSV data into a dictionary
description_dict = {}
with open('./../../data/wikidata5m_descriptions/wikidata5m_2sentences.tsv', 'r') as file:
    for line in file:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            description_dict[parts[0]] = parts[1]

# Function to append descriptions to JSONL data
def append_descriptions_to_jsonl(input_jsonl, output_jsonl):
    with open(input_jsonl, 'r') as infile, open(output_jsonl, 'w') as outfile:
        for line in infile:
            data = json.loads(line.strip())
            # Append sub_id description if available
            if data['sub_id'] in description_dict:
                data['sub_desc'] = description_dict[data['sub_id']]
            else:
                data['sub_desc'] = "No description available"

            # Append obj_id description if available
            if data['obj_id'] in description_dict:
                data['obj_desc'] = description_dict[data['obj_id']]
            else:
                data['obj_desc'] = "No description available"
            
            # Write updated JSON to the new file
            json.dump(data, outfile, ensure_ascii=False)
            outfile.write('\n')

# Example usage
append_descriptions_to_jsonl(
    './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc.jsonl',
    './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc_wikipedia.jsonl'
)


**Append the descriptions to the training datasets in csv**

In [7]:
# Load JSONL data and create mappings for descriptions
sub_desc_dict = {}
obj_desc_dict = {}

with open('./../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc_wikipedia.jsonl', 'r') as jsonl_file:
    for line in jsonl_file:
        record = json.loads(line.strip())
        sub_desc_dict[record['sub_value']] = record.get('sub_desc', "No description available")
        obj_desc_dict[record['obj_value']] = record.get('obj_desc', "No description available")

# Function to append descriptions to CSV data
def append_descriptions_to_csv(input_csv, output_csv):
    with open(input_csv, 'r') as infile, open(output_csv, 'w', newline='') as outfile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames + ['subject_desc', 'object_desc']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            subject = row['subject']
            object = row['object']
            # Append descriptions using the dictionaries
            row['subject_desc'] = sub_desc_dict.get(subject, "No description available")
            row['object_desc'] = obj_desc_dict.get(object, "No description available")
            writer.writerow(row)

# Example usage
append_descriptions_to_csv('./../../data/dataset/wikidata5m_42k_test.csv', './../../data/dataset/wikidata5m_42k_desc_test.csv')

In [22]:
triples_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

entity_ids = set()
with open(triples_path, 'r') as triples_file:
    for line in triples_file:
        parts = line.strip().split('\t')
        if len(parts) >= 3:  # Ensure the line is valid
            entity_ids.add(parts[0])  # Add subject entity ID
            entity_ids.add(parts[2])

print(f"Number of unique entities: {len(entity_ids)}")

Number of unique entities: 61085


## Similarity scores using Wembedder

In [5]:
def get_similarity_score(sub_id, obj_id):
    api_url = f"https://wembedder.toolforge.org/api/similarity/{sub_id}/{obj_id}"
    try:
        response = requests.get(api_url, timeout=10)  # Adding timeout for robustness
        response.raise_for_status()
        data = response.json()
        return data.get('similarity')
    except Exception:
        #print(f"Error fetching similarity for {sub_id} and {obj_id}: {e}")
        return None

# Function to process a single triple, to be used with parallel execution
def process_single_triple(triple):
    sim_score = get_similarity_score(triple['sub_id'], triple['obj_id'])
    if sim_score is not None:
        triple['sim_score'] = sim_score
    return triple

# Updated function to process triples in parallel
def process_triples_parallel(input_file, output_file, max_workers=10):
    triples_to_process = []
    with open(input_file, 'r', encoding='utf-8') as infile:
        triples_to_process = [json.loads(line) for line in infile]

    # Using ThreadPoolExecutor to parallelize the API requests
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all triples for processing
        future_to_triple = {executor.submit(process_single_triple, triple): triple for triple in triples_to_process}
        
        with open(output_file, 'w', encoding='utf-8') as outfile:
            for future in tqdm(as_completed(future_to_triple)):
                triple = future_to_triple[future]
                try:
                    updated_triple = future.result()
                    json.dump(updated_triple, outfile, ensure_ascii=False)
                    outfile.write('\n')
                except Exception as exc:
                    print(f'{triple} generated an exception: {exc}')

# Adjusted file paths for testing
input_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k.jsonl'
output_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k_sim.jsonl'

process_triples_parallel(input_file_path, output_file_path)

299943it [4:04:27, 20.45it/s]


In [7]:
# count how many similarities are not None
output_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k_sim.jsonl'

similarity_scores = []
with open(output_file_path, 'r', encoding='utf-8') as infile:
    for line in infile:
        triple = json.loads(line)
        if 'sim_score' in triple:
            similarity_scores.append(triple['sim_score'])

print(f"Number of similarity scores: {len(similarity_scores)}")

Number of similarity scores: 42422


In [19]:
# count the number of entities (subject, object)
file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc.jsonl'

entity_ids = set()
with open(file, 'r', encoding='utf-8') as infile:
    for line in infile:
        triple = json.loads(line)
        entity_ids.add(triple['sub_id'])
        entity_ids.add(triple['obj_id'])

print(f"Number of unique entities: {len(entity_ids)}")

Number of unique entities: 55344


In [7]:
def filter_jsonl(input_file_path, output_file_path):
    """
    Reads a JSONL file and filters out entries without a 'sim_score' attribute.
    Writes the filtered data to a new JSONL file.
    """
    with open(input_file_path, 'r', encoding='utf-8') as input_file, \
         open(output_file_path, 'w', encoding='utf-8') as output_file:
        for line in input_file:
            try:
                data = json.loads(line)
                # Check if 'sim_score' is in the data
                if 'sim_score' in data:
                    json.dump(data, output_file, ensure_ascii=False)
                    output_file.write('\n')
            except json.JSONDecodeError:
                print("Warning: Skipped invalid JSON line.")
                continue

# Example usage:
input_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_300k_sim.jsonl'
output_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim.jsonl'  # Desired path for the output file

filter_jsonl(input_file_path, output_file_path)

In [21]:
# saving the dataset with similarity scores to a csv file only for triples with similarity scores 
input_file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc.jsonl'
output_file_path = './../../data/dataset/wikidata5m_42k.csv'

with open(input_file_path, 'r') as jsonl_file, open(output_file_path, 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['subject', 'object', 'similarity'])

    for line in jsonl_file:
        data = json.loads(line)
        if 'sim_score' in data:
            # Extract the 'sub_value', 'obj_value', and 'sim_score'
            subject = data['sub_value']
            object_ = data['obj_value']
            similarity = data['sim_score']

            csv_writer.writerow([subject, object_, similarity])


In [4]:
# Load the dataset
df = pd.read_csv('./../../data/dataset/wikidata5m_42k.csv')

# Split the dataset into training and a temporary set with an 80:20 ratio
train, temp = train_test_split(df, test_size=0.2, random_state=42)

# Split the temporary set into validation and test sets with a 50:50 ratio
valid, test = train_test_split(temp, test_size=0.5, random_state=42)

# Save the training, validation, and test sets to new CSV files
train.to_csv('./../../data/dataset/wikidata5m_42k_train.csv', index=False)
valid.to_csv('./../../data/dataset/wikidata5m_42k_valid.csv', index=False)
test.to_csv('./../../data/dataset/wikidata5m_42k_test.csv', index=False)

# Print the sizes of the training, validation, and test sets
print(f"Training set size: {len(train)}")
print(f"Validation set size: {len(valid)}")
print(f"Test set size: {len(test)}")

Training set size: 33937
Validation set size: 4242
Test set size: 4243


## Analysis of the dataset

In [9]:
def count_unique_entities(file_path: str):
    unique_ids_sub = set()
    unique_ids_obj = set()
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            unique_ids_sub.add(data['sub_id'])
            unique_ids_obj.add(data['obj_id'])
    all_entities = unique_ids_sub.union(unique_ids_obj)
    return len(all_entities)

def count_unique_relations(file_path: str):
    unique_relations = set()
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            unique_relations.add(data['pred_id'])
    return len(unique_relations)

def count_unique_objects(file_path: str):
    unique_objects = set()
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            unique_objects.add(data['obj_id'])
    return len(unique_objects)

def count_unique_subjects(file_path: str):
    unique_subjects = set()
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            unique_subjects.add(data['sub_id'])
    return len(unique_subjects)

def count_unique_triples(file_path: str):
    unique_triples = set()
    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            unique_triples.add((data['sub_id'], data['pred_id'], data['obj_id']))
    return len(unique_triples)

def plot_relation_distribution(file_path):
    relation_counts = {}  # Dictionary to count occurrences of each relation

    with open(file_path, 'r') as file:
        for line in file:
            data = json.loads(line)
            # Increment the count of the relation in the dictionary
            if data['pred_id'] in relation_counts:
                relation_counts[data['pred_id']] += 1
            else:
                relation_counts[data['pred_id']] = 1

    counts = list(relation_counts.values())

    plt.figure(figsize=(10, 8))
    plt.hist(counts, bins=20, color='green', edgecolor='black')  # Configure number of bins as needed
    plt.xlabel('Number of Triples per Relation')
    plt.ylabel('Number of Relations')
    plt.title('Histogram of Relation Occurrences in Knowledge Graph')
    plt.grid(True)
    plt.show()

def plot_original_relation_distribution(file_path):
    relation_counts = {}  # Dictionary to count occurrences of each relation

    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split('\t')  # Split the line into parts by tab
            relation_id = parts[1]  # Get the relation id (the second element)
            # Increment the count of the relation in the dictionary
            if relation_id in relation_counts:
                relation_counts[relation_id] += 1
            else:
                relation_counts[relation_id] = 1

    counts = list(relation_counts.values())

    plt.figure(figsize=(10, 8))
    plt.hist(counts, bins=20, color='red', edgecolor='black')  # Configure number of bins as needed
    plt.xlabel('Number of Triples per Relation')
    plt.ylabel('Number of Relations')
    plt.title('Histogram of Relation Occurrences in Knowledge Graph')
    plt.grid(True)
    plt.show()

# Example usage
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_42k_sim_desc.jsonl'
print(f"Number of unique entities: {count_unique_entities(file_path)}")

Number of unique entities: 55344
